# Markowitz Mean-Variance Optimization

### Wealth Management Portfolio Optimization — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), introductory optimisation concepts, and basic understanding of wealth management and portfolio optimisation terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Explain the optimisation problem and how it maps to a real wealth management and portfolio optimisation decision.
- Implement the optimisation workflow and interpret the solution in portfolio or business terms.
- Assess the sensitivity of the optimal solution to input assumptions and identify when the model may mislead.

**Estimated time:** 35–45 minutes

**Collection context:** Portfolio optimisation, asset allocation, risk management, and investment strategy for wealth management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** Modern Portfolio Theory (MPT), introduced by Harry Markowitz, is the foundation of institutional investing. It helps investors maximize return for a given level of risk by diversifying across assets that don't move perfectly together.

**Input data used:** Historical (synthetic) prices/returns of Stocks, Bonds, Real Estate, and Gold.

**What we calculate:** Portfolio weights (how much % to invest in each asset).

**Math method used:** Quadratic Programming (Optimization) using `scipy.optimize`.

**Learning goal:** Understand how "Diversification is the only free lunch in finance."

**Primary stakeholders:** wealth advisors, portfolio managers, investment committee members, and financial planners.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import minimize

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Build Synthetic Asset Returns

We create 5 years of monthly returns for four different asset classes with realistic risk/reward profiles.

In [ ]:
assets = ['Stocks', 'Bonds', 'Real Estate', 'Gold']
n_assets = len(assets)
n_months = 60

# Annualized Expected Returns and Volatility
exp_returns_ann = np.array([0.10, 0.04, 0.07, 0.05])
volatility_ann = np.array([0.18, 0.05, 0.12, 0.15])

# Convert to monthly
exp_returns_mo = exp_returns_ann / 12
volatility_mo = volatility_ann / np.sqrt(12)

# Correlation Matrix (Assets move differently)
corr_matrix = np.array([
    [1.0, 0.1, 0.6, 0.0],
    [0.1, 1.0, 0.2, 0.3],
    [0.6, 0.2, 1.0, 0.1],
    [0.0, 0.3, 0.1, 1.0]
])

# Covariance Matrix
cov_mo = np.outer(volatility_mo, volatility_mo) * corr_matrix

# Generate Returns
returns = RNG.multivariate_normal(exp_returns_mo, cov_mo, n_months)
df_returns = pd.DataFrame(returns, columns=assets)

print("Sample of Monthly Returns:")
print(df_returns.head())

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

We define functions to calculate portfolio return, volatility, and Sharpe ratio (return per unit of risk).

In [ ]:
def portfolio_stats(weights, returns_df):
    weights = np.array(weights)
    port_return = np.sum(returns_df.mean() * weights) * 12
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns_df.cov() * 12, weights)))
    sharpe_ratio = (port_return - 0.02) / port_vol # Assume 2% risk-free rate
    return port_return, port_vol, sharpe_ratio

def neg_sharpe(weights, returns_df):
    return -portfolio_stats(weights, returns_df)[2]

def port_volatility(weights, returns_df):
    return portfolio_stats(weights, returns_df)[1]

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: equal-weight (1/N) portfolio ---
equal_weights = np.array([1 / n_assets] * n_assets)
eq_ret, eq_vol, eq_sharpe = portfolio_stats(equal_weights, df_returns)
print(f"Equal-weight baseline  Return: {eq_ret:.2%}  Vol: {eq_vol:.2%}  Sharpe: {eq_sharpe:.2f}")
print("Optimisation should deliver a better risk-adjusted return.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1}) # Weights must sum to 100%
bounds = tuple((0, 1) for _ in range(n_assets)) # No short selling (weights between 0 and 1)
initial_weights = [1/n_assets] * n_assets

opt_results = minimize(
    neg_sharpe, 
    initial_weights, 
    args=(df_returns,), 
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

best_weights = opt_results.x
res_return, res_vol, res_sharpe = portfolio_stats(best_weights, df_returns)

print("Optimal Weights:")
for asset, weight in zip(assets, best_weights):
    print(f"{asset}: {weight:.2%}")

print(f"\nExpected Annual Return: {res_return:.2%}")
print(f"Expected Volatility: {res_vol:.2%}")
print(f"Sharpe Ratio: {res_sharpe:.2f}")

In [ ]:
n_simulations = 2000
sim_returns = []
sim_vols = []
sim_sharpes = []

for _ in range(n_simulations):
    w = RNG.random(n_assets)
    w /= np.sum(w)
    r, v, s = portfolio_stats(w, df_returns)
    sim_returns.append(r)
    sim_vols.append(v)
    sim_sharpes.append(s)

plt.figure(figsize=(10, 6))
plt.scatter(sim_vols, sim_returns, c=sim_sharpes, cmap='viridis', alpha=0.3)
plt.colorbar(label='Sharpe Ratio')
plt.scatter(res_vol, res_return, color='#E76F51', marker='*', s=200, label='Max Sharpe Ratio Portfolio')
plt.xlabel('Annualized Volatility (Risk)')
plt.ylabel('Annualized Return')
plt.title('Markowitz Efficient Frontier (Simulated)')
plt.legend()
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart, scatter plot titled "Markowitz Efficient Frontier (Simulated)" with Annualized Volatility (Risk) on the x-axis and Annualized Return on the y-axis. The chart highlights spatial separation or clustering patterns in the data.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Risk vs. Reward:** Notice how the red star is at the top-left edge of the cloud. It offers the highest return for its level of risk.
2. **Diversification:** The optimal portfolio likely holds a mix of assets (e.g., mostly Stocks for growth, but Bonds and Gold to lower the risk).
3. **Real World:** Wealth managers use this to create "Model Portfolios" (Conservative, Moderate, Aggressive) based on a client's risk tolerance.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: compare risk-appetite portfolios ---
print("Operational demonstration — portfolios for different client profiles:\n")

cons = minimize(port_volatility, initial_weights, args=(df_returns,),
                method='SLSQP', bounds=bounds, constraints=constraints)
cons_ret, cons_vol, cons_sharpe = portfolio_stats(cons.x, df_returns)

print("Conservative (minimum volatility):")
for a, w in zip(assets, cons.x):
    if w > 0.01:
        print(f"  {a}: {w:.1%}")
print(f"  Return: {cons_ret:.2%}  Vol: {cons_vol:.2%}  Sharpe: {cons_sharpe:.2f}\n")

print("Aggressive (maximum Sharpe):")
for a, w in zip(assets, best_weights):
    if w > 0.01:
        print(f"  {a}: {w:.1%}")
print(f"  Return: {res_return:.2%}  Vol: {res_vol:.2%}  Sharpe: {res_sharpe:.2f}")

print("\nA wealth advisor would adjust these for client-specific constraints.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end wealth management and portfolio optimisation workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Optimisation models may suggest portfolios unsuitable for clients with different risk tolerances or time horizons.
- Model assumptions about return distributions may not hold during market crises.
- Fiduciary duty requires that model outputs support, not replace, professional judgement.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in wealth management and portfolio optimisation?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.